# 05 - Agent Query App

This is the main notebook for demo.

Lightweight single agent system:
1. Understand the question
2. Route to the right tool
3. Retrieve runbook context or generate SQL
4. Return the final answer

In [0]:
%run ./01_ingest_parse_pdf

In [0]:
%run ./02_chunk_prepare_index

In [0]:
%run ./04_tools_layer

## Main agent function

In [0]:
def ask_agent(question: str):
    route = route_question(question)

    print(f"Selected tool: {route['tool']}")
    print(f"Reason: {route['reason']}")

    # 1. SQL only
    if route["tool"] == "generate_validation_sql":
        sql_text = generate_validation_sql(question)

        print("\nGenerated SQL:\n")
        print(sql_text)

        print("\nNotes:\n")
        print("- Replace source_table and target_table with real table names")
        print("- Adjust key/column names as needed")
        return

    # 2. Validation guidance + SQL
    if route["tool"] == "search_and_sql":
        chunks = search_runbooks(question, num_results=3)
        sql_text = generate_validation_sql(question)

        print("\nRetrieved context:\n")
        print_chunks(chunks)

        prompt = build_guidance_prompt(question, chunks)
        safe_prompt = prompt.replace("'", "''")

        response = spark.sql(f"""
        SELECT ai_query(
          '{LLM_ENDPOINT}',
          '{safe_prompt}'
        ) AS answer
        """)

        print("\nGenerated SQL:\n")
        print(sql_text)

        print("\nFinal Guidance:\n")
        display(response)
        return

    # 3. Troubleshooting
    if route["tool"] == "search_and_summarize":
        chunks = search_runbooks(question, num_results=3)

        print("\nRetrieved context:\n")
        print_chunks(chunks)

        prompt = build_troubleshooting_prompt(question, chunks)
        safe_prompt = prompt.replace("'", "''")

        response = spark.sql(f"""
        SELECT ai_query(
          '{LLM_ENDPOINT}',
          '{safe_prompt}'
        ) AS answer
        """)

        print("\nFinal Answer:\n")
        display(response)
        return

    print("No route matched.")

## Example 1 - Recovery & Validation

In [0]:
ask_agent("My pipeline failed due to authentication issues. After fixing it, what validations should I run before marking it successful?")

## Example 2 - Data Integrity Validation

In [0]:
ask_agent("How do I ensure data integrity between source and target after pipeline execution?")

## Example 3 - Null Spike After Pipeline

In [0]:
ask_agent("After rerunning a pipeline, I see many null values in a critical column. How do I validate and troubleshoot this?")

##Example 4 - Duplicate Data Issue

In [0]:
ask_agent("How do I detect and fix duplicate records introduced during pipeline processing?")

##Example 5 - Schema Drift Scenario

In [0]:
ask_agent("Pipeline failed due to schema mismatch after source system changes. What steps should I take?")

##Example 6 - Incremental Load Validation

In [0]:
ask_agent("How do I validate that my incremental load processed only new and updated records correctly?")

##Example 7 - SQL Error Handling

In [0]:
ask_agent("My pipeline is failing due to conversion errors. How should I handle this in SQL?")